In [0]:
%fs ls /Volumes/gizmobox/landing/operational_data/customers

In [0]:
select * from json.`/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json`

In [0]:
select _metadata.file_path as source,
    *
    from json.`/Volumes/gizmobox/landing/operational_data/customers/customers_*.json`

In [0]:
create or replace view gizmobox.bronze.v_customers
as
select     *,_metadata.file_path as file_path
    from json.`/Volumes/gizmobox/landing/operational_data/customers/customers_*.json`

In [0]:
select * from gizmobox.bronze.v_customers;

In [0]:
create or replace temporary view tv_customers
as
select     *,_metadata.file_path as file_path
    from json.`/Volumes/gizmobox/landing/operational_data/customers/customers_*.json`

In [0]:
select * from tv_customers;

In [0]:
create or replace global temporary view gtv_customers
as
select     *,_metadata.file_path as file_path
    from json.`/Volumes/gizmobox/landing/operational_data/customers/customers_*.json`

In [0]:
select * from global_temp.gtv_customers;

## Complex JSON Extract

In [0]:
use gizmobox.bronze;
select * from json.`/Volumes/gizmobox/landing/operational_data/orders/orders_*.json`

In [0]:

create or replace view gizmobox.bronze.v_orders as
select * from text.`/Volumes/gizmobox/landing/operational_data/orders/orders_*.json`

In [0]:
select * from gizmobox.bronze.v_orders;

In [0]:
select current_catalog(),current_schema()

## Unstructured Data Extract

In [0]:
create or replace view gizmobox.bronze.v_memberships as
select * from binaryFile.`/Volumes/gizmobox/landing/operational_data/memberships/*/*.png`

In [0]:
use gizmobox.bronze;
select * from v_memberships;

In [0]:
select * from read_files('/Volumes/gizmobox/landing/operational_data/addresses/',
                format => 'csv',
                header => 'true',
                delimiter => '\t'
                )

In [0]:
create or replace view gizmobox.bronze.v_addresses as
select * from read_files('/Volumes/gizmobox/landing/operational_data/addresses/',
                format => 'csv',
                header => 'true',
                delimiter => '\t'
                )

In [0]:
select * from v_addresses;

## External Table creation

In [0]:
create table if not exists gizmobox.bronze.payments
(payment_id integer, order_id integer,payment_timestamp timestamp, payment_status integer, payment_method string)
using csv
options (header = 'true', delimiter = ",")
location 'abfss://gizmobox@dbudemyextdatalake.dfs.core.windows.net/landing/external_data/payments'

In [0]:
describe extended gizmobox.bronze.payments

In [0]:
select * from gizmobox.bronze.payments;

## Refresh the table to reflec the file changes in table

In [0]:
refresh table gizmobox.bronze.payments;

## JDBC connection lakehouse federation
### jdbc:sqlserver://gizmobox-server2406.database.windows.net:1433;database=gizmobox-db

In [0]:
create connection azure_sql_gizmobox_conn_sql
type sqlserver
options (
    host "gizmobox-server2406.database.windows.net",
    port "1433",
    user "dbadmin",
    password "Irula@24061809"
)

In [0]:
create foreign catalog if not exists azure_gizmobox_catalog_sql using connection azure_sql_gizmobox_conn_sql
options (database 'gizmobox-db');

In [0]:
select * from azure_gizmobox_catalog_sql.dbo.refunds